In [140]:
import ast
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from collections import Counter

In [141]:
df = pd.read_csv('/media/prince/5A4E832F4E83034D/Movie recomender/Train_v2/clean_data.csv')

In [142]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   genres                4803 non-null   object 
 1   keywords              4803 non-null   object 
 2   original_language     4803 non-null   object 
 3   overview              4800 non-null   object 
 4   popularity            4803 non-null   float64
 5   production_companies  4803 non-null   object 
 6   runtime               4803 non-null   float64
 7   tagline               3959 non-null   object 
 8   title                 4803 non-null   object 
 9   vote_average          4803 non-null   float64
 10  vote_count            4803 non-null   float64
 11  f_genres              4803 non-null   object 
 12  genre_names           4803 non-null   object 
 13  f_keywords            4803 non-null   object 
 14  keywords_names        4803 non-null   object 
dtypes: float64(4), object

In [143]:
df['companies'] = df['production_companies'].apply(ast.literal_eval)

In [144]:
df['companies_names'] = df['companies'].apply(lambda x: [d['name'] for d in x])

all_companies = []
demo = []
for sub in df['companies_names']:
    for company in sub:
        company = company.strip().lower()
        all_companies.append(company)
        demo.append(company)
all_companies = list(set(all_companies))
df['companies_list'] = df['companies_names']

In [145]:
company_counts = Counter(all_companies)
common_companies = [item[0] for item in company_counts.most_common(200)]

cleaned = []
for row in df['companies_names']:
    new_list = []
    for key in row:
        k = key.strip().lower()  # clean it before comparing
        if k in common_companies:
            new_list.append(k)
        else:
            new_list.append('other')
    new_list = list(set(new_list))
    cleaned.append(new_list)

df['companies_names'] = cleaned
df['companies_names'].head(20)

0                                   [other]
1                                   [other]
2                                   [other]
3                                   [other]
4                                   [other]
5                                   [other]
6                                   [other]
7                                   [other]
8                                   [other]
9                                   [other]
10                                  [other]
11                                  [other]
12                                  [other]
13                                  [other]
14                                  [other]
15                                  [other]
16                                  [other]
17                                  [other]
18    [other, parkes/macdonald productions]
19                                  [other]
Name: companies_names, dtype: object

In [146]:
other_companies = df['companies_names'].apply(lambda x: x == ['other'])
other_companies_count = other_companies.sum()
other_companies_count

np.int64(3985)

In [147]:
df['companies_list'].to_csv('companies_.csv')

In [148]:
df['companies_list'].head()

0    [Ingenious Film Partners, Twentieth Century Fo...
1    [Walt Disney Pictures, Jerry Bruckheimer Films...
2                     [Columbia Pictures, Danjaq, B24]
3    [Legendary Pictures, Warner Bros., DC Entertai...
4                               [Walt Disney Pictures]
Name: companies_list, dtype: object

In [149]:
demo_count = Counter(demo)
least_common_item_tuple = min(demo_count.items(), key=lambda item: item)
mostlist = demo_count.most_common()

least_to_most_list = mostlist[::-1]

# 3. Get the first 10 items (the 10 least common ones)
#    [0:10] means start at index 0 and stop before index 10.
least_10_elements = least_to_most_list[4500:4510]
least_10_elements
# mostlist[0], least_10_elements
# len(demo)
len(mostlist)

5006

In [150]:
count_3 = sum(1 for company, count in mostlist if count >= 3)
count_5 = sum(1 for company, count in mostlist if count >= 5)
count_10 = sum(1 for company, count in mostlist if count >= 10)


big_studio = {company for company, count in mostlist if count >= 10}
indie = {company for company, count in mostlist if count <=2}

print("\nCompanies appearing ≥ 3 times:", count_3)
print("Companies appearing ≥ 5 times:", count_5)
print("Companies appearing ≥ 10 times:", count_10)

# 4️⃣ Find the index of the last company that has ≥3 occurrences
last_3_index = None
for i, (company, count) in enumerate(mostlist):
    if count < 3:
        last_3_index = i - 1
        break

print("\nLast index where company appears ≥3 times:", last_3_index)
print("Last company with ≥3 appearances:", mostlist[last_3_index])
print("Total companies with ≥3 appearances:", last_3_index + 1)


Companies appearing ≥ 3 times: 838
Companies appearing ≥ 5 times: 462
Companies appearing ≥ 10 times: 188

Last index where company appears ≥3 times: 837
Last company with ≥3 appearances: ('duplass brothers productions', 3)
Total companies with ≥3 appearances: 838


In [151]:
cleaned = []

for row in df['companies_list']:       # loop each movie row
    tags = []

    for c in row:                     # loop each company in that movie
        if c.strip().lower() in big_studio:
            tags.append('big_studio') # mark big if company in big list

    if len(row) > 1:                  # if more than 1 company → multi
        tags.append('multi_studio')

    if len(row) == 1 and 'big_studio' not in tags:
        tags.append('indie')          # 1 company + not big → indie

    if not tags:                      # if still empty → force indie
        tags.append('indie')

    cleaned.append(list(set(tags)))   # remove duplicates & save

df['companies_categories'] = cleaned  # store result in dataframe


In [152]:
df['companies_categories']

0       [big_studio, multi_studio]
1       [big_studio, multi_studio]
2       [big_studio, multi_studio]
3       [big_studio, multi_studio]
4                     [big_studio]
                   ...            
4798                  [big_studio]
4799                       [indie]
4800                [multi_studio]
4801                       [indie]
4802                [multi_studio]
Name: companies_categories, Length: 4803, dtype: object

In [153]:
df.to_csv("data_veiw.csv", index=False)

In [154]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   genres                4803 non-null   object 
 1   keywords              4803 non-null   object 
 2   original_language     4803 non-null   object 
 3   overview              4800 non-null   object 
 4   popularity            4803 non-null   float64
 5   production_companies  4803 non-null   object 
 6   runtime               4803 non-null   float64
 7   tagline               3959 non-null   object 
 8   title                 4803 non-null   object 
 9   vote_average          4803 non-null   float64
 10  vote_count            4803 non-null   float64
 11  f_genres              4803 non-null   object 
 12  genre_names           4803 non-null   object 
 13  f_keywords            4803 non-null   object 
 14  keywords_names        4803 non-null   object 
 15  companies            

In [155]:
df2 = df.drop(columns= ['keywords', 'genres', 'production_companies', 'f_genres', 'f_keywords', 'companies', 'companies_names', 'companies_list'])

In [156]:
df2.head(10)

,original_language,overview,popularity,runtime,tagline,title,vote_average,vote_count,genre_names,keywords_names,companies_categories
0,en,"In the 22nd century, a paraplegic Marine is di...",150.437577,162.0,Enter the World of Pandora.,Avatar,7.2,11800.0,"['Action', 'Adventure', 'Fantasy', 'Science Fi...","['other', 'romance', 'battle', 'future', 'alie...","[big_studio, multi_studio]"
1,en,"Captain Barbossa, long believed to be dead, ha...",139.082615,169.0,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500.0,"['Adventure', 'Fantasy', 'Action']","[""love of one's life"", 'other', 'ship', 'after...","[big_studio, multi_studio]"
2,en,A cryptic message from Bond’s past sends him o...,107.376788,148.0,A Plan No One Escapes,Spectre,6.3,4466.0,"['Action', 'Adventure', 'Crime']","['other', 'spy', 'based on novel', 'sequel', '...","[big_studio, multi_studio]"
3,en,Following the death of District Attorney Harve...,112.312950,165.0,The Legend Ends,The Dark Knight Rises,7.6,9106.0,"['Action', 'Crime', 'Drama', 'Thriller']","['other', 'secret identity', 'superhero', 'ter...","[big_studio, multi_studio]"
4,en,"John Carter is a war-weary, former military ca...",43.926995,132.0,"Lost in our world, found in another.",John Carter,6.1,2124.0,"['Action', 'Adventure', 'Science Fiction']","['other', 'alien', 'based on novel', 'princess...",[big_studio]
5,en,The seemingly invincible Spider-Man goes up ag...,115.699814,139.0,The battle within.,Spider-Man 3,5.9,3576.0,"['Fantasy', 'Action', 'Adventure']","[""love of one's life"", 'other', 'superhero', '...","[big_studio, multi_studio]"
6,en,When the kingdom's most wanted-and most charmi...,48.681969,100.0,They're taking adventure to new lengths.,Tangled,7.4,3330.0,"['Animation', 'Family']","['other', 'magic', 'musical', 'hostage', 'hors...","[big_studio, multi_studio]"
7,en,When Tony Stark tries to jumpstart a dormant p...,134.279229,141.0,A New Age Has Come.,Avengers: Age of Ultron,7.3,6767.0,"['Action', 'Adventure', 'Science Fiction']","['other', 'based on comic book', 'superhero', ...","[big_studio, multi_studio]"
8,en,"As Harry begins his sixth year at Hogwarts, he...",98.885637,153.0,Dark Secrets Revealed,Harry Potter and the Half-Blood Prince,7.4,5293.0,"['Adventure', 'Fantasy', 'Family']","['witch', 'other', 'magic']","[big_studio, multi_studio]"
9,en,Fearing the actions of a god-like Super Hero l...,155.790452,151.0,Justice or revenge,Batman v Superman: Dawn of Justice,5.7,7004.0,"['Action', 'Adventure', 'Fantasy']","['revenge', 'other', 'based on comic book', 's...","[big_studio, multi_studio]"


In [157]:
df2.to_csv('final-cleaned-data.csv')